In [1]:
import sys
sys.path.insert(0, "..")

import mne
from pathlib import Path
from src.preprocessing.loader import load_raw
from src.preprocessing.filter import apply_filters
from src.preprocessing.epoching import make_epochs
from src.preprocessing.artifacts import mark_bad_channels, reject_by_amplitude, log_rejection
from pipeline import load_config

cfg = load_config("../config.yaml")
print("Imports OK")

Imports OK


In [2]:
USE_SAMPLE_DATA = True  # flip to False (and set the path below) to run against a local recording

if USE_SAMPLE_DATA:
    sample_path = mne.datasets.sample.data_path()
    raw_path = sample_path / "MEG" / "sample" / "sample_audvis_raw.fif"
else:
    raw_path = Path(cfg["paths"]["raw_data"]) / "your_recording.fif"

raw = load_raw(str(raw_path))
# exclude=[] keeps channels already flagged bad in the recording's own metadata (e.g. EEG 053) —
# pick_types' default (exclude='bads') would silently drop them before this notebook's own
# bad-channel marking step below gets a chance to demonstrate on them.
raw_eeg = raw.copy().pick_types(eeg=True, meg=False, stim=True, eog=False, exclude=[])

bp = cfg["preprocessing"]["bandpass"]
raw_eeg = apply_filters(raw_eeg, l_freq=bp["l_freq"], h_freq=bp["h_freq"], notch_freq=cfg["preprocessing"]["notch_freq"])

print(f"Channels: {len(raw_eeg.ch_names)}")

Opening raw data file C:\Users\ke725\mne_data\MNE-sample-data\MEG\sample\sample_audvis_raw.fif...


    Read a total of 3 projection items:


        PCA-v1 (1 x 102)  idle


        PCA-v2 (1 x 102)  idle


        PCA-v3 (1 x 102)  idle


...

- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)


- Upper passband edge: 40.00 Hz


- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)


- Filter length: 1983 samples (3.302 s)


Channels: 69


In [3]:
raw_eeg.plot(duration=10, n_channels=30, title="EEG before bad-channel marking")

Using matplotlib as 2D backend.


C:\Users\ke725\Documents\eeg-project\.venv\Lib\site-packages\mne\viz\utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


<MNEBrowseFigure size 800x800 with 4 Axes>

**Bad-channel check**

The plot above uses MNE's interactive Qt browser, which renders live and isn't captured as a
static image in this notebook — scroll through it yourself when you run this cell to confirm
what's below. `EEG 053` is the channel MNE's own sample-dataset tutorials consistently flag as
bad (visible as a channel with a distinctly larger, noisier trace than its neighbors). Marking
it bad here follows that precedent — but treat it as a claim to verify against your own view of
the plot, not a fact to take on faith.

In [4]:
raw_eeg = mark_bad_channels(raw_eeg, bad_channels=["EEG 053"])

print(f"Bad channels marked: {raw_eeg.info['bads']}")

Marked 1 channel(s) as bad: ['EEG 053']
Bad channels marked: ['EEG 053']


In [5]:
pp = cfg["preprocessing"]
epochs = make_epochs(raw_eeg, tmin=pp["epoch_tmin"], tmax=pp["epoch_tmax"])
print(epochs)

Finding events on: STI 014


320 events found on stim channel STI 014


Event IDs: [ 1  2  3  4  5 32]


Not setting metadata


320 matching events found


...

Applying baseline correction (mode: mean)


0 projection items activated


Using data from preloaded Raw for 320 events and 601 original time points ...


0 bad epochs dropped


<Epochs | 320 events (all good), -0.2 – 0.799 s (baseline -0.2 – 0 s), ~104.2 MiB, data loaded,
 'auditory/left': 72
 'auditory/right': 73
 'visual/left': 73
 'visual/right': 71
 'smiley': 15
 'buttonpress': 16>


In [6]:
epochs_clean = reject_by_amplitude(epochs, peak_to_peak_thresh={"eeg": 150e-6})

    Rejecting  epoch based on EEG : ['EEG 003', 'EEG 007']


    Rejecting  epoch based on EEG : ['EEG 001', 'EEG 002', 'EEG 003', 'EEG 004', 'EEG 005', 'EEG 006', 'EEG 007', 'EEG 015', 'EEG 016']


    Rejecting  epoch based on EEG : ['EEG 001', 'EEG 002', 'EEG 003', 'EEG 004', 'EEG 005', 'EEG 006', 'EEG 007', 'EEG 012', 'EEG 013', 'EEG 014', 'EEG 015', 'EEG 016']


    Rejecting  epoch based on EEG : ['EEG 001', 'EEG 002', 'EEG 003', 'EEG 004', 'EEG 006', 'EEG 007']


    Rejecting  epoch based on EEG : ['EEG 001', 'EEG 002', 'EEG 003', 'EEG 006', 'EEG 007']


...

    Rejecting  epoch based on EEG : ['EEG 001', 'EEG 002', 'EEG 003', 'EEG 004', 'EEG 006', 'EEG 007']


    Rejecting  epoch based on EEG : ['EEG 001', 'EEG 002', 'EEG 003', 'EEG 007']


    Rejecting  epoch based on EEG : ['EEG 001', 'EEG 002', 'EEG 003', 'EEG 007']


    Rejecting  epoch based on EEG : ['EEG 001', 'EEG 002', 'EEG 003', 'EEG 007']


31 bad epochs dropped


In [7]:
log_rejection(epochs, epochs_clean)

Epochs before rejection : 320
Epochs after rejection  : 289
Dropped                 : 31 (9.7%)
Kept                    : 289 (90.3%)


**What I Observed in M4**

**Bad channel:** `EEG 053` — this channel is already flagged bad in the sample dataset's own
metadata (`raw.info['bads']` before any picking), which is why `pick_types` needed
`exclude=[]` above to keep it around long enough for this notebook to mark it explicitly
itself. Once marked, `mark_bad_channels` correctly refused an invalid name (verified
separately in `tests/preprocessing/test_artifacts.py`), and confirmed here with
`Marked 1 channel(s) as bad: ['EEG 053']`.

**Rejection threshold:** 150 µV peak-to-peak on EEG channels (`{"eeg": 150e-6}`), the
value `reject_by_amplitude` defaults to.

**Before/after counts:** 320 epochs before rejection, 289 kept, 31 dropped (9.7%).

**Interpretation:**
- A ~10% drop rate is on the high side of "reasonable" for a clean sample recording, but
  not alarming — the sample dataset is a real (not simulated) recording with genuine blink
  and muscle artifacts, and 150 µV is a fairly strict threshold by ERP-analysis convention.
- The rejection log rarely names `EEG 053` as a cause — because it's marked bad, MNE's
  `drop_bad` excludes it from the amplitude check entirely rather than penalizing epochs for
  its (likely much larger) amplitude swings. The channels driving most rejections instead
  are `EEG 001`–`EEG 007`, frontal sites consistent with eye-blink artifact, which M5's ICA
  is specifically built to separate out instead of discarding whole epochs for.
- This is exactly the gap M3's epoch-drop count couldn't see: `summarize_epochs` in M3
  reported 0 dropped epochs (320 found, 320 kept) because every trial fit the time window —
  but 31 of those 320 "kept" epochs contain artifact amplitude large enough to distort an
  averaged ERP if left in. M4's job is catching what M3 structurally can't.